# Lekcia 09 - Návrhový vzor metakognície


## Nastavenie

Tento notebook demonštruje návrhový vzor Metacognition s použitím Microsoft Agent Framework.

**Požiadavky:**
- Nasadenie Azure OpenAI nakonfigurované cez premenné prostredia
- Azure CLI autentifikované (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Čo je metakognícia?

Metakognícia je **premýšľanie o premýšľaní**. V kontexte AI agentov to znamená vytváranie agentov, ktorí dokážu:

- **Reflektovať** svoje vlastné výstupy a proces uvažovania
- **Zisťovať chyby** a zotaviť sa elegantne namiesto tichého zlyhania
- **Hodnotiť** či sú ich odpovede úplné a užitočné
- **Prispôsobiť** svoju stratégiu, keď počiatočný prístup nefunguje (napr. prechod na záložný systém)

Metakognitívny agent nielen odpovedá na otázky — sleduje vlastný výkon a priebežne sa prispôsobuje.


## Primárne a záložné nástroje

Bežný vzorec metakognície je **náhradná stratégia**. Agent najprv vyskúša primárny nástroj; ak zlyhá (napr. chyba 404), agent rozpozná zlyhanie a transparentne prejde na záložný nástroj.

Toto odráža systémy v reálnom svete, kde primárne služby môžu byť nedostupné a agenti si musia problém sami diagnostikovať skôr, ako zvolia alternatívnu cestu.

Nižšie definujeme dva nástroje na vyhľadávanie letov:
- **Primárny** — pokrýva Paríž, Tokio a Barcelona
- **Záložný** — pokrýva Berlín, Sydney a New York


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Sebareflektujúci agent s obnovou po chybe

Agent nižšie je inštruovaný, aby najprv vyskúšal primárny letový systém, rozpoznal zlyhania a transparentne sa vrátil na záložný systém. Po každej odpovedi krátko zhodnotí, či plne odpovedal na otázku používateľa.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Vzor seba-hodnotenia

Ďalším aspektom metakognície je **seba-hodnotenie**: samostatný agent (alebo ten istý agent pri druhom prebehu) posudzuje odpoveď z hľadiska úplnosti, presnosti a užitočnosti.

Nižšie vytvoríme agenta `ResponseEvaluator`, ktorý hodnotí odpovede cestovného agenta podľa troch kritérií.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Zhrnutie

V tejto lekcii ste sa naučili, ako vytvárať **metakognitívne agenty** pomocou Microsoft Agent Framework:

- **Sebareflexia**: Agenti, ktorí sledujú svoje vlastné uvažovanie a priehľadne komunikujú, čo sa stalo.
- **Obnova pri chybách s náhradnými zdrojmi**: Vzor primárneho + záložného nástroja, kde agent zistí zlyhania (napr. chyby 404) a automaticky vyskúša alternatívny zdroj.
- **Seba-evaluácia**: Samostatný hodnotiaci agent, ktorý hodnotí odpovede z hľadiska úplnosti, presnosti a užitočnosti.

Tieto vzory robia agentov robustnejšími, priehľadnejšími a dôveryhodnejšími — kľúčové vlastnosti pre nasadenie do produkčného prostredia.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vyhlásenie o vylúčení zodpovednosti:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa usilujeme o presnosť, berte prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho originálnom jazyku by mal byť považovaný za autoritatívny zdroj. Pri kritických informáciách sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo mylné výklady vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
